# Lesson 4.6: Time Series in Pandas

**Analyst:** Meron Welderufael
**Date:** 11/06/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Produce the cohort churn breakdown Hannah asked for
this morning, plus supporting monthly-signup trend and YoY comparison.
Deliverable for end-of-week.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('finflow-enriched-200k.csv',
                parse_dates = ['signup_date', 'last_login_date', 'churn_date'])

print(f'Loaded {len(df):,} rows')
print(f'Signup date range: {df['signup_date'].min()} to {df['signup_date'].max()}')
print(f'Columns: {df.shape[1]}')

Loaded 200,000 rows
Signup date range: 2024-10-22 00:00:00 to 2026-10-19 00:00:00
Columns: 22


## Question 1: The cohort-by-tier cross-cut.

In [2]:
df['signup_cohort'] = df['signup_date'].dt.to_period('M')
#Inspect
print(df[['customer_id', 'signup_date', 'marketing_name', 'signup_cohort']].head())



  customer_id signup_date marketing_name signup_cohort
0    FF-00001  2026-03-11   Starter Plan       2026-03
1    FF-00002  2025-12-06   Starter Plan       2025-12
2    FF-00003  2026-04-04   Starter Plan       2026-04
3    FF-00004  2025-10-25   Starter Plan       2025-10
4    FF-00005  2026-02-25   Professional       2026-02


In [3]:
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

cohort_by_tier = df.groupby(['signup_cohort', 'marketing_name']).agg(
    customer_count = ('customer_id', 'count'),
    churn_rate_pct = ('active', churn_rate_pct)
).unstack(level='marketing_name').round(2)

print(cohort_by_tier.shape)
print(cohort_by_tier.head(5))
print('---(rows omitted)---')
print(cohort_by_tier.tail(5))

(25, 6)
                customer_count                            churn_rate_pct  \
marketing_name Enterprise Plus Professional Starter Plan Enterprise Plus   
signup_cohort                                                              
2024-10                      8           20           32            0.00   
2024-11                     86          243          445            4.65   
2024-12                    167          501         1111            3.59   
2025-01                    270          802         1573            3.33   
2025-02                    347          981         1972            4.90   

                                          
marketing_name Professional Starter Plan  
signup_cohort                             
2024-10               10.00        15.62  
2024-11               12.35        26.74  
2024-12               12.57        24.21  
2025-01               15.34        23.27  
2025-02               14.27        25.05  
---(rows omitted)---
                cu

In [4]:
cohort_by_tier.to_csv('cohort-by-tier-breakdown.csv',index=False)
print('Saved cohort-by-tier-breakdown.csv ')

Saved cohort-by-tier-breakdown.csv 


The Starter Plan has the highest cohort churn rate, generally around 25%, while Enterprise Plus has the lowest, generally around 5%. Professional falls between the two. This is consistent with how we designed the synthetic dataset, with target churn rates of 25% for Starter and 5% for Enterprise.

## Question 2: Cohort size trend.

In [5]:
# Set signup_date as index for resampling
df_indexed = df.set_index('signup_date')

signups_by_quarter = df_indexed.resample('QE').size()
print(signups_by_quarter)
print(f'Total quarters: {len(signups_by_quarter)}')

signup_date
2024-12-31     2613
2025-03-31    10402
2025-06-30    18682
2025-09-30    27407
2025-12-31    35765
2026-03-31    43499
2026-06-30    42685
2026-09-30    18362
2026-12-31      585
Freq: QE-DEC, dtype: int64
Total quarters: 9


In [6]:
# quarterly change in signups
quarter_prev = signups_by_quarter.shift(1)
quarter_change = signups_by_quarter - quarter_prev
quarter_pct = (quarter_change/quarter_prev) * 100

comparison_quarter = pd.DataFrame({
    'current': signups_by_quarter,
    'prev': quarter_prev,
    'change': quarter_change,
    'pct_change': quarter_pct
})
print(comparison_quarter)

             current     prev    change  pct_change
signup_date                                        
2024-12-31      2613      NaN       NaN         NaN
2025-03-31     10402  2613.00   7789.00      298.09
2025-06-30     18682 10402.00   8280.00       79.60
2025-09-30     27407 18682.00   8725.00       46.70
2025-12-31     35765 27407.00   8358.00       30.50
2026-03-31     43499 35765.00   7734.00       21.62
2026-06-30     42685 43499.00   -814.00       -1.87
2026-09-30     18362 42685.00 -24323.00      -56.98
2026-12-31       585 18362.00 -17777.00      -96.81


Signups increased consistently from Q1 2025 through Q1 2026, although the percentage growth gradually slowed. Q2 2026 showed a small decline of 1.87%, followed by a larger decline of 56.98% in Q3 2026. However, the first and last quarters are incomplete, so those figures should be interpreted cautiously. The recent decline is worth investigating further.

## Question 3: A six-month rolling cohort churn average.

In [7]:
cohort_churn = pd.read_csv('cohort-churn-by-signup-month.csv',
                          parse_dates= [ 'signup_cohort'])
print(cohort_churn.shape)

(25, 4)


In [8]:

cohort_churn['churn_rate_pct_6mo_rolling'] = cohort_churn['churn_rate_pct'].rolling(window=6).mean()
print(cohort_churn)

   signup_cohort  customer_count  churn_rate_pct  avg_monthly_fee  \
0     2024-10-01              60           11.67            41.86   
1     2024-11-01             774           19.77            37.27   
2     2024-12-01            1779           19.00            33.37   
3     2025-01-01            2645           18.83            35.35   
4     2025-02-01            3300           19.73            35.81   
5     2025-03-01            4457           21.05            35.25   
6     2025-04-01            5178           20.01            35.42   
7     2025-05-01            6359           20.49            35.29   
8     2025-06-01            7145           20.20            34.87   
9     2025-07-01            8195           19.77            34.72   
10    2025-08-01            9281           20.36            34.20   
11    2025-09-01            9931           19.25            34.02   
12    2025-10-01           11213           20.04            35.83   
13    2025-11-01           11651  

FinFlow's cohort churn has remained relatively flat over the dataset's history. The six-month rolling average ranges from 18.34% to 20.24%, indicating only small fluctuations rather than a clear upward or downward trend. Overall, churn appears stable at approximately 20%.

## Question 4: A short written summary.

**Looking at the cohort-by-tier table from Question 1, do you see different churn rates for the same cohort across different tiers? Which tier has the highest cohort churn rates? Which has the lowest?**

The Starter plan has the highest cohort churn rate, while the Enterprise tier has the lowest. This is consistent with how we designed the synthetic dataset, where we targeted a 25% churn rate for Starter customers and 5% for Enterprise customers.

**Looking at the six-month rolling cohort churn from Question 3, is FinFlow's cohort churn trending up, down, or flat over the dataset's history?**

FinFlow's cohort churn appears relatively flat over the dataset's history. The six-month rolling churn rates range from 18.34% to 20.24%, showing only small fluctuations rather than a clear upward or downward trend.

**One specific number from your analysis that you'd put in the email back to Hannah.**

Some of the month-over-month percentage changes are above 100%, indicating significant increases in signups. However, some of the more recent months show negative percentage changes. I would highlight this as something worth investigating to understand whether the recent decline is a seasonal pattern or a potential issue affecting customer acquisition.